In [2]:
import rpy2.robjects.packages as rpackages
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import pandas as pd
import tqdm

import os
import sqlite3
import platform

pandas2ri.activate()

rpackages.importr('DBI')
rpackages.importr('lme4')
rpackages.importr('DT')


rpy2.robjects.packages.Package as a <module 'DT'>

In [1]:
#Linear models - Oh and Schuler - 

In [3]:

def extract_model_summary(model_name, summary_name):

    # 1. Fixed Effects
    fixed_effects = ro.r(f'as.data.frame({summary_name}$coefficients)')
    fixed_effects_df = pandas2ri.rpy2py(fixed_effects)
    #print(fixed_effects_df)
    fixed_effects_df.columns = ['Estimate', 'Std. Error', 't value'] #, 'Pr(>|t|)']

    # 2. Random Effects
    random_effects = ro.r(f'as.data.frame(VarCorr({model_name}))')
    random_effects_df = pandas2ri.rpy2py(random_effects)
    #print("\nRandom Effects (Variance and Std. Dev by Group):\n", random_effects_df)

    # 3. Residuals
    residuals = ro.r(f'as.data.frame({summary_name}$residuals)')
    residuals_df = pandas2ri.rpy2py(residuals)
    #print("\nResiduals:\n", residuals_df)

    # 4. Model Fit Statistics
    aic = ro.r(f'AIC({model_name})')[0]
    bic = ro.r(f'BIC({model_name})')[0]
    log_likelihood = ro.r(f'logLik({model_name})')[0]
    warnings = ro.r('warnings()') 
    fit_stats_df = pd.DataFrame({
        'AIC': [aic],
        'BIC': [bic],
        'Log-Likelihood': [log_likelihood],
        'Warnings': [warnings]
    })

    # 5. Variance-Covariance Matrix of Random Effects
    var_cov_matrix = ro.r(f'as.data.frame({summary_name}$varcor)')
    var_cov_matrix_df = pandas2ri.rpy2py(var_cov_matrix)
    
    return fixed_effects_df, random_effects_df, fit_stats_df, var_cov_matrix_df
def predict_rt(fit_name):
    ro.r(f'''
    baseline_df$PredictedLogRT <- predict({fit_name})
    ''')

    # # Extract the predicted RT values from the R data frame
    # predicted_rt = ro.r(f'as.data.frame(baseline_df)')
    
    #Column names of interest is "RTUID", "WorkerID", "StoryWordID", "LogRT", predicted_RT"

    ro.r('''predicted_rt <- baseline_df[, c("RTUID", "WorkerID", "StoryWordID", "LogRT", "PredictedLogRT")]''')

    #Print Head of the predicted RT values
    
    # Convert the predicted RT values to a pandas DataFrame
    predicted_rt_df = pandas2ri.rpy2py(ro.r('predicted_rt'))
    
    return predicted_rt_df

def execute_r_lmer_model(fit_formula, data_frame_name, fit_name, fit_summary_name):
    ro.r(f'''
    {fit_name} <- lmer({fit_formula}, data={data_frame_name}, REML=F)
    {fit_summary_name} <- summary({fit_name})
    ''')
    
    fixed_effects, random_effects, fit_stats, var_cov_matrix = extract_model_summary(fit_name, fit_summary_name)

    predicted_rt_df = predict_rt(fit_name)
    return fixed_effects, random_effects, fit_stats, var_cov_matrix,  predicted_rt_df


#Given a model id, load the dataset for it in r, transform the data and run the model and return the results

#Replace SPRTNaturalStories with RTDundeeCorpus
def fit_surprisal_model(filter_model_id):
    ro.r(f'''
           RESULTS_DB_PATH <- "/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/results/results.db"
            results_db <- dbConnect(RSQLite::SQLite(), RESULTS_DB_PATH)

            baseline_df <- dbGetQuery(results_db,"WITH FilteredModel AS (
            SELECT StoryWordID, SurprisalScore
            FROM ModelSurprisalScores
            WHERE ModelID = {filter_model_id}
            )

            SELECT RTDundeeCorpus.RTUID, 
            RTDundeeCorpus.WorkerID, 
            RTDundeeCorpus.StoryWordID, 
            RTDundeeCorpus.GazeDuration,
            
            WordDetails.Word as WordCategory, 
            WordDetails.CharacterLength, 
            WordDetails.WordUID as WordCategoryID,
            WordDetails.LogFrequencies as LogFrequencies,

            Story.POSTag as POSTag,
            
            FilteredModel.SurprisalScore as SurprisalScore
            
            FROM RTDundeeCorpus
            JOIN Story on RTDundeeCorpus.StoryWordID = Story.StoryWordID 
            JOIN WordDetails on WordDetails.WordUID = Story.WordUID 
            JOIN FilteredModel on FilteredModel.StoryWordID = RTDundeeCorpus.StoryWordID
            
            WHERE RTDundeeCorpus.IgnoreRow=0

                
            ")
            '''
            )
    
    
    ro.r('''
        baseline_df$LogRT <- log(baseline_df$GazeDuration)
        
        baseline_df$WorkerID <- as.factor(baseline_df$WorkerID)
        baseline_df$WordCategoryID <- as.factor(baseline_df$WordCategoryID)
        baseline_df$POSTag <- as.factor(baseline_df$POSTag)
        baseline_df$CharacterLength_c <- scale(baseline_df$CharacterLength)

        #Should I scale Log Frequencies(?)
        baseline_df$LogFrequencies_c <- scale(baseline_df$LogFrequencies)

        baseline_df$SurprisalScore_c <- scale(baseline_df$SurprisalScore)
        data_size <- nrow(baseline_df)
        ''')
    
    data_frame_name = "baseline_df"

    fit_name_m = "fit_model"
    fit_summary_name_m = "fit_model_summary"

    fixed_effects_m, random_effects_m, fit_stats_m, var_cov_matrix_m, predicted_rt_df = execute_r_lmer_model(fit_formula_m, "baseline_df", fit_name_m, fit_summary_name_m)

    return fixed_effects_m, random_effects_m, fit_stats_m, var_cov_matrix_m, predicted_rt_df


def compile_results_as_df(model_id, fixed_effects_df, random_effects_df, fit_stats_df, var_cov_matrix_df, fit_stats_b):
    
    data_size = ro.r('data_size')[0]
    results_df_row = {
        "ModelID" : model_id,
        "condition_model_formula" : fit_formula_m,
        "Log-Likelihood" : fit_stats_df['Log-Likelihood'].values[0],
        "Coefficient for Surprisal Score" : fixed_effects_df.loc['SurprisalScore_c', 'Estimate'],
        "Delta Log-Likelihood" : fit_stats_df['Log-Likelihood'].values[0] - fit_stats_b['Log-Likelihood'].values[0],
        "AIC" : fit_stats_df['AIC'].values[0],
        "BIC" : fit_stats_df['BIC'].values[0],
        "Data Size" : data_size,
        "baseline_model_formula" : fit_formula_b,
        'Fixed Effects': fixed_effects_df.to_html(),
        'Random Effects': random_effects_df.to_html(),
        'Variance-Covariance Matrix': var_cov_matrix_df.to_html(),
    }
        
    #results_df = pd.DataFrame(results_df_row, index=[0])

    return results_df_row



def append_or_overwrite_csv(file_path, new_row):
    # Read the existing CSV file
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        # If the file does not exist, create a new DataFrame
        df = pd.DataFrame(columns=new_row.keys())
    
    # Check if the row already exists
    # This assumes that the DataFrame has a unique identifier column called 'id'
    if 'ModelID' in new_row:  # Ensure that there's a unique identifier
        existing_row_index = df[df['ModelID'] == new_row['ModelID']].index
        
        if not existing_row_index.empty:
            # If the row exists, update it
            df.loc[existing_row_index[0]] = new_row  # Update the first matching row
        else:
            # If the row does not exist, append it
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    else:
        print("New row must contain a unique identifier under the key 'ModelID'.")

    # Save the updated DataFrame back to CSV
    df.to_csv(file_path, index=False)



In [ ]:

if "pop-os" in platform.node():
    ROOT = r"/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/"
else:
    ROOT = r'/gpfs/home4/athamma/repo/ss-llm/nanoGPT/'#Given a set of details, return a dataframe compiled with the details

RESULTS_ROOT = os.path.join(ROOT, "results")
SQL_DB = os.path.join(RESULTS_ROOT, "results.db")


def create_connection_cursor(db_file):
    """
    Create a database connection to the SQLite database specified by the db_file

    Args:
        db_file (str): database file

    Returns:
        Connection object or None
    """
    conn = sqlite3.connect(db_file)
    c = conn.cursor()
    return conn, c

conn, c = create_connection_cursor(SQL_DB)

MODEL_LIST_QUERY = "SELECT ModelID from ModelSurprisalScores"

MODEL_LIST_QUERY = '''
SELECT DISTINCT ModelSurprisalScores.ModelID, Model.OutputFolderName, Model.BatchSize, Model.Dataset, Model.Seed, Model.MaskType FROM ModelSurprisalScores
JOIN Model on Model.ModelID = ModelSurprisalScores.ModelID
WHERE Model.NumLayers = 6 
AND ((Model.EchoicMemory=10 AND Model.MaskType="exponential_new" AND Model.MaskDecayRate=2) OR (Model.MaskType="Non" AND Model.CurriculumLearning=False)) 
AND Model.Dataset in ("babylm_full_bpe_8k", "babylm_full_bpe_100M_8k")  
AND Model.ModelID not in (5496427, 8456913)
ORDER BY Seed, BatchSize, Dataset, MaskType
'''

# MODEL_LIST_QUERY = '''
# SELECT DISTINCT ModelSurprisalScores.ModelID, Model.OutputFolderName, Model.BatchSize, Model.Dataset, Model.Seed, Model.MaskType FROM ModelSurprisalScores
# JOIN Model on Model.ModelID = ModelSurprisalScores.ModelID
# WHERE Model.NumLayers = 6 AND Model.MaskType="exponential_new" AND Model.MaskDecayRate=2 
# AND Model.ModelID not in (5496427, 8456913)
# ORDER BY Seed, BatchSize, Dataset, MaskType

# '''

model_id_list = pd.read_sql_query(MODEL_LIST_QUERY, conn)['ModelID'].unique().tolist()

#model_id_list = model_id_list[:1]

print(len(model_id_list), model_id_list)
fit_stats_b = None


62 [8465733, 8465085, 6892214, 6839425, 8117279, 8117319, 8465604, 8465082, 6892212, 6839403, 8111923, 8111939, 8465611, 8465091, 6892220, 6839430, 8111928, 8111942, 8465607, 8465086, 6892216, 6839426, 8111925, 8111941, 8465610, 8465090, 6892219, 6839429, 8117282, 8117325, 8096895, 8465077, 6892222, 6681944, 8098216, 8111938, 8464992, 8456915, 8465605, 8465084, 6892213, 6839424, 8111924, 8111940, 8465609, 8465089, 6892218, 6839428, 8117281, 8117322, 8465612, 8465093, 6892221, 6839431, 8117283, 8117326, 8465608, 8465087, 6892217, 6839427, 8117280, 8117321]


In [5]:

data_frame_name = "baseline_df"
fit_name_b = "fit_baseline"
fit_summary_name_b = "fit_summary_baseline"

# fixed_effects_b, random_effects_b, fit_stats_b, var_cov_matrix_b = execute_r_lmer_model(fit_formula_b, data_frame_name, fit_name_b, fit_summary_name_b)

predicted_variable = "LogGazeDuration"  # LogGazeDuration or LogFirstFixationDuration or LogGoPastTime or LogRT

fit_formula_b0 = f"LogRT ~ CharacterLength_c + (1 | WorkerID) + (1 | POSTag)"
fit_formula_b = f"LogRT ~ CharacterLength_c + LogFrequencies_c + (1 | WorkerID) + (1 | POSTag)"
fit_formula_m = f"LogRT ~ CharacterLength_c + LogFrequencies_c + SurprisalScore_c + (1 | WorkerID) + (1 | POSTag)"




# write_path = "surprisal_analysis_results_oshdundee.csv"

#results_df = pd.DataFrame()

def verify_model_already_processed(write_path):
    conn = sqlite3.connect(write_path)
    c = conn.cursor()
    c.execute("SELECT DISTINCT ModelID FROM RTModelPredict WHERE StoryWordID > 10256")
    rows = c.fetchall()
    conn.close()
    return [row[0] for row in rows]


current_processed_model_list = verify_model_already_processed(SQL_DB)
for model_id in tqdm.tqdm(model_id_list):
    if model_id in current_processed_model_list:
        continue
    try:    
        fixed_effects_m, random_effects_m, fit_stats_m, var_cov_matrix_m, predict_rt_df_m  = fit_surprisal_model(model_id)
        predict_rt_df_m["ModelID"] = model_id
        predict_rt_df_m = predict_rt_df_m[["ModelID", "RTUID", "WorkerID", "StoryWordID", "LogRT", "PredictedLogRT"]]
        predict_rt_df_m.to_sql("RTModelPredict", conn, if_exists="append", index=False)
        
    except Exception as e:
        print(f"Error for model id {model_id}: {e}")
        continue
        



  0%|          | 0/62 [00:00<?, ?it/s]R[write to console]: In addition: 
R[write to console]: Warning messages:

R[write to console]: 1: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages

R[write to console]: 2: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages

R[write to console]: 3: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages

  2%|▏         | 1/62 [00:08<08:45,  8.62s/it]R[write to console]: In addition: 
R[write to console]: Warning message:

R[write to console]: call dbDisconnect() when finished working with a connection

[]

In [9]:
fixed_effects_m, random_effects_m, fit_stats_m, var_cov_matrix_m, predict_rt_df = fit_surprisal_model(model_id_list[0])



In [ ]:
predict_rt_df

,RTUID,WorkerID,StoryWordID,LogRT,PredictedLogRT
1,2,1,10258,5.049856,5.467524
2,51503,2,10258,5.049856,5.715854
3,103004,3,10258,5.886104,5.428200
4,412010,9,10258,4.682131,5.476885
5,3,1,10259,5.424950,5.504130
...,...,...,...,...,...
195791,154502,3,61756,5.093750,5.284118
195792,206003,4,61756,5.455321,5.423100
195793,515009,10,61756,5.537334,5.428245
195794,51501,1,61757,5.953243,5.281437


In [15]:
test_df = pd.read_sql("SELECT * FROM RTDundeeCorpus", conn)

test_df

,RTUID,StoryWordID,WorkerID,GazeDuration,FirstFixationDuration,GoPastTime,WordSkipped,WordFixationCount,IgnoreRow
0,1,10257,sa,216.0,216.0,216.0,0,1,1
1,2,10258,sa,156.0,156.0,156.0,0,1,0
2,3,10259,sa,227.0,227.0,401.0,0,2,0
3,4,10260,sa,0.0,0.0,0.0,1,0,1
4,5,10261,sa,187.0,187.0,525.0,0,3,0
...,...,...,...,...,...,...,...,...,...
515005,515006,61753,sj,290.0,110.0,290.0,0,2,0
515006,515007,61754,sj,0.0,0.0,0.0,1,0,1
515007,515008,61755,sj,336.0,181.0,336.0,0,2,0
515008,515009,61756,sj,254.0,254.0,254.0,0,1,0
